# Explore power-law statistics results

Notebook for inspecting Clauset outputs under `results/pl_stats/`.
This first pass focuses on **ACCRE** with **fixed `xmin=2`** (`real/xmin02`).

Use the `process-mining-rfcs` Jupyter kernel.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import powerlaw

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_attachments
from utils.powerlaw.plotting import _plot_alternative_overlay
from utils.rfc import extract_frequency_counts

from analyses.powerlaw_statistics.analysis_helpers.paths import stack_raw_summaries

RESULTS_ROOT = REPO_ROOT / "results" / "pl_stats" / "real" / "xmin02"
RAW_RESULTS = RESULTS_ROOT / "raw_results"
LOG_NAME = "ACCRE"
MODEL = "lower_bounded_power_law"
COMPARE_CONCEPTS = ("n4", "n5")

DISPLAY_COLS = [
    "log_name",
    "concept",
    "n_occurrences",
    "n_types",
    "xmin",
    "n_fitted_types",
    "best_other_distribution",
    "alpha",
    "KS_D",
    "gof_p",
    "n_success",
    "n_failed",
    "gof_kind",
    "Comment",
]

## Summary table (`real/xmin02`)

The GOF $p$-value ($p_{\mathrm{GOF}}$) **increases a lot between $n=4$ and $n=5$**:
power-law plausibility jumps from rejected ($p < 0.1$) to comfortably accepted.

In [ ]:
summary = stack_raw_summaries(RAW_RESULTS)
summary = summary.loc[summary["log_name"].eq(LOG_NAME)].copy()
summary["Comment"] = ""
table = summary.loc[:, [c for c in DISPLAY_COLS if c in summary.columns or c == "Comment"]]
table

## Empirical CCDF: $n=4$ vs $n=5$ (same scale)

Side-by-side log–log complementary CDFs with the fitted power-law (and best alternative when available).
Axes limits are shared so the jump in GOF is visually comparable.
Plotting reuses `utils.powerlaw.plotting._plot_alternative_overlay` (same overlay as compose).

In [ ]:
def load_frequencies(concept: str):
    row = summary.loc[summary["concept"].eq(concept)].iloc[0]
    path = Path(str(row["input_path"]))
    if not path.is_absolute():
        path = REPO_ROOT / path
    frequencies = extract_frequency_counts(load_attachments(path))
    xmin = float(row["xmin"]) if pd.notna(row["xmin"]) else None
    alt = row.get("best_other_distribution")
    alt_name = None if alt is None or (isinstance(alt, float) and pd.isna(alt)) else str(alt)
    fit = powerlaw.Fit(
        frequencies,
        discrete=True,
        xmin=xmin,
        xmax=None,
        verbose=False,
        parameter_ranges={"alpha": [0.0, 4.0]},
    )
    return frequencies, fit, alt_name, row


panels = {c: load_frequencies(c) for c in COMPARE_CONCEPTS}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, concept in zip(axes, COMPARE_CONCEPTS):
    _freq, fit, alt_name, row = panels[concept]
    fit.plot_ccdf(ax=ax, color="red", linewidth=1.8, label="Observed")
    _plot_alternative_overlay(fit, ax, kind="ccdf", alternative_name=alt_name)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(
        f"{LOG_NAME} {concept} (xmin={int(row['xmin'])}, "
        f"gof_p={float(row['gof_p']):.4f})"
    )
    ax.set_xlabel("Frequency")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

axes[0].set_ylabel("CCDF")
fig.tight_layout()
plt.show()